# Retained Experiments

목표: 지금까지 실험 중 실제로 남겨둘 가치가 있었던 전처리만 한 노트북에서 다시 확인합니다.

포함 실험:
- `team_baseline_binary`
- `constant + duplicate 제거`
- `minimum frequency >= 2`

정리 기준:
- 제출 점수 방어용 1순위는 `baseline`
- 성능 손실 없이 정리 가능한 전처리는 보조 후보로 유지
- full benchmark 기준 baseline을 넘지 못한 복잡한 feature engineering은 제외


In [ ]:
from __future__ import annotations

from dataclasses import dataclass
import os
from pathlib import Path
import sys
import tempfile
from typing import Any

os.environ.setdefault('MPLCONFIGDIR', str(Path(tempfile.gettempdir()) / 'matplotlib-member-b-retained'))

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
from IPython.display import display
from scipy import sparse
from sklearn.base import BaseEstimator, TransformerMixin, clone
from sklearn.pipeline import Pipeline
from tqdm.auto import tqdm

pd.set_option('display.max_columns', 200)
sns.set_theme(style='whitegrid')


def find_project_root(start: Path) -> Path:
    for path in [start, *start.parents]:
        if (path / 'requirements.txt').exists() and (path / 'experiments').exists():
            return path
    raise FileNotFoundError('저장소 루트를 찾지 못했습니다. 저장소 루트에서 JupyterLab을 실행하세요.')


ROOT = find_project_root(Path.cwd())
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

from common.preprocessing_benchmark import run_preprocessing_benchmark
from common.starter_preprocess import make_baseline_preprocessor

print('project root:', ROOT)


In [ ]:
@dataclass(frozen=True)
class NotebookConfig:
    id_col: str = 'ID'
    target_col: str = 'SUBCLASS'
    wt_value: str = 'WT'


CONFIG = NotebookConfig()
EXPERIMENT_MODE = 'benchmark'  # 'fast' | 'benchmark'
FAST_SAMPLE_PER_CLASS = 30
RUN_MODELS = ['logistic']
USE_CONFIRMATION = False

train_df = pd.read_csv(ROOT / 'data' / 'raw' / 'train.csv')
feature_columns = [c for c in train_df.columns if c not in {CONFIG.id_col, CONFIG.target_col}]


def prepare_experiment_dataset(train_df: pd.DataFrame) -> pd.DataFrame:
    if EXPERIMENT_MODE == 'benchmark':
        return train_df.copy()

    sampled_parts = []
    for _, class_frame in train_df.groupby(CONFIG.target_col, sort=True):
        sample_size = min(len(class_frame), FAST_SAMPLE_PER_CLASS)
        sample_size = max(sample_size, 5)
        sampled_parts.append(class_frame.sample(n=sample_size, random_state=42))
    return pd.concat(sampled_parts, ignore_index=True)


experiment_train_df = prepare_experiment_dataset(train_df)

print('train shape:', train_df.shape)
print('experiment rows:', experiment_train_df.shape[0])
print('feature count:', len(feature_columns))


In [ ]:
def hash_signature(series: pd.Series) -> tuple[int, ...]:
    return tuple(pd.util.hash_pandas_object(series, index=False).tolist())


class BasicFilteringBinaryEncoder(BaseEstimator, TransformerMixin):
    def __init__(self, *, wt_value: str = 'WT') -> None:
        self.wt_value = wt_value

    def fit(self, X: pd.DataFrame, y: Any = None) -> 'BasicFilteringBinaryEncoder':
        del y
        self.feature_names_in_ = np.asarray(X.columns, dtype=object)
        raw = X.reindex(columns=self.feature_names_in_)

        nunique = raw.nunique(dropna=True)
        self.constant_columns_ = nunique[nunique <= 1].index.tolist()

        seen = {}
        duplicate_cols = []
        for col in self.feature_names_in_:
            sig = hash_signature(raw[col])
            if sig in seen and raw[col].equals(raw[seen[sig]]):
                duplicate_cols.append(col)
            else:
                seen[sig] = col
        self.duplicate_columns_ = duplicate_cols

        removable = set(self.constant_columns_) | set(self.duplicate_columns_)
        self.keep_columns_ = [col for col in self.feature_names_in_ if col not in removable]
        return self

    def transform(self, X: pd.DataFrame):
        aligned = X.reindex(columns=self.keep_columns_).fillna(self.wt_value).astype(str)
        normalized = aligned.apply(lambda col: col.str.strip())
        binary = normalized.ne(self.wt_value).astype('float32')
        return sparse.csr_matrix(binary.to_numpy(dtype=np.float32))

    def get_feature_names_out(self, input_features: Any = None) -> np.ndarray:
        del input_features
        return np.asarray(self.keep_columns_, dtype=object)


class MinimumFrequencyBinaryEncoder(BaseEstimator, TransformerMixin):
    def __init__(self, *, wt_value: str = 'WT', min_frequency: int = 2) -> None:
        self.wt_value = wt_value
        self.min_frequency = min_frequency

    def fit(self, X: pd.DataFrame, y: Any = None) -> 'MinimumFrequencyBinaryEncoder':
        del y
        self.feature_names_in_ = np.asarray(X.columns, dtype=object)
        aligned = X.reindex(columns=self.feature_names_in_).fillna(self.wt_value).astype(str)
        normalized = aligned.apply(lambda col: col.str.strip())
        binary = normalized.ne(self.wt_value).astype('float32')
        counts = binary.sum(axis=0)
        self.keep_columns_ = counts[counts >= self.min_frequency].index.tolist()
        return self

    def transform(self, X: pd.DataFrame):
        aligned = X.reindex(columns=self.keep_columns_).fillna(self.wt_value).astype(str)
        normalized = aligned.apply(lambda col: col.str.strip())
        binary = normalized.ne(self.wt_value).astype('float32')
        return sparse.csr_matrix(binary.to_numpy(dtype=np.float32))

    def get_feature_names_out(self, input_features: Any = None) -> np.ndarray:
        del input_features
        return np.asarray(self.keep_columns_, dtype=object)


EXPERIMENTS = [
    ('team_baseline_binary', '공용 baseline: WT를 0/1 binary로 변환', make_baseline_preprocessor()),
    ('basic_filter_constant_duplicate', 'constant + duplicate 제거 후 binary', Pipeline([('basic_filter', BasicFilteringBinaryEncoder())])),
    ('min_freq_ge_2', 'fold 내부 mutation count >= 2 gene만 유지', Pipeline([('min_freq', MinimumFrequencyBinaryEncoder(min_frequency=2))])),
]


In [ ]:
summary_rows: list[dict[str, Any]] = []
fold_metric_frames: list[pd.DataFrame] = []

for experiment_name, description, preprocessor in tqdm(EXPERIMENTS, desc='Retained experiments'):
    for model_name in tqdm(RUN_MODELS, desc=f'{experiment_name} models', leave=False):
        result = run_preprocessing_benchmark(
            experiment_train_df,
            clone(preprocessor),
            experiment_id=experiment_name,
            preprocessing_name=description,
            model=model_name,
            confirmation=USE_CONFIRMATION,
            target_column=CONFIG.target_col,
            id_column=CONFIG.id_col,
            verbose=False,
        )
        summary_rows.append({
            'experiment': experiment_name,
            'description': description,
            'model': model_name,
            'oof_macro_f1': result.summary['oof_f1_macro_mean'],
            'oof_accuracy': result.summary['oof_accuracy_mean'],
            'elapsed_seconds': result.summary['elapsed_seconds'],
            'feature_count_mean': float(result.fold_metrics['feature_count'].mean()),
        })
        fold_df = result.fold_metrics.copy()
        fold_df['experiment'] = experiment_name
        fold_df['model'] = model_name
        fold_metric_frames.append(fold_df)

summary_results = pd.DataFrame(summary_rows)
baseline_scores = (
    summary_results[summary_results['experiment'] == 'team_baseline_binary'][['model', 'oof_macro_f1']]
    .rename(columns={'oof_macro_f1': 'baseline_macro_f1'})
)
summary_results = summary_results.merge(baseline_scores, on='model', how='left')
summary_results['delta_vs_baseline'] = summary_results['oof_macro_f1'] - summary_results['baseline_macro_f1']
summary_results = summary_results.sort_values(['model', 'oof_macro_f1'], ascending=[True, False]).reset_index(drop=True)
display(summary_results)

fold_metric_results = pd.concat(fold_metric_frames, ignore_index=True)
display(fold_metric_results.groupby(['experiment', 'model'], as_index=False).agg(
    fold_macro_f1_mean=('f1_macro', 'mean'),
    fold_macro_f1_std=('f1_macro', 'std'),
    elapsed_seconds_mean=('elapsed_seconds', 'mean'),
))


In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(16, 5))

sns.barplot(data=summary_results, x='experiment', y='oof_macro_f1', hue='model', ax=axes[0], palette='Blues')
axes[0].set_title('OOF Macro F1')
axes[0].tick_params(axis='x', rotation=45)

sns.barplot(data=summary_results, x='experiment', y='delta_vs_baseline', hue='model', ax=axes[1], palette='Oranges')
axes[1].axhline(0, color='black', linewidth=1, linestyle='--')
axes[1].set_title('Delta vs Team Baseline')
axes[1].tick_params(axis='x', rotation=45)

plt.tight_layout()
plt.show()


## 현재 해석

- `team_baseline_binary`
  - full benchmark 기준 최고 성능 기준선
- `basic_filter_constant_duplicate`
  - 성능 손실 없이 feature 수를 줄이는 정리형 후보
- `min_freq_ge_2`
  - benchmark 기준 baseline보다 아주 소폭 낮지만, 경량화 후보로는 유지 가능

실전 제출 점수 방어가 목적이면 `team_baseline_binary`를 우선 사용하고,
속도나 메모리 이득이 필요할 때만 나머지 두 후보를 보조안으로 고려합니다.
